# 260716 R - questions and work

* want to take random data set of milan's and match it up with our functions
* want to fix and understand the KK functions and apply them to our calculations and compare


In [1]:
%matplotlib widget 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.ticker as mticker

In [2]:
#3: Basic functions defined for use
def cos(th):
    return np.cos(th*np.pi/180) 
def costht(ni, nt, ki, kt, thi):
    return(np.sqrt(1-((ni+1j*ki)/(nt+1j*kt)*sin(thi))**2)) 
def arcsin(ratio):
    return np.arcsin(ratio)*180/np.pi 
def sin(th):
    return np.sin(th*np.pi/180) 

mu0 = 4*np.pi*10**-7

#4: Returns transmitted angle from incident angle. It gives you theta t from theta i. 
def snells(ni, nt, ki, kt, thi):
    return(arcsin((ni+1j*ki)/(nt+1j*kt)*sin(thi)))

#5: General form of fresnel coefficients for s polarized light (perpindicular electric field) 
def r_s(ni, nt, ki, kt, thi, mui=4*np.pi*10**-7, mut=4*np.pi*10**-7):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    return(((ni+1j*ki)/mui*cos_thi-(nt+1j*kt)/mut*cos_tht)/((ni+1j*ki)/mui*cos_thi+(nt+1j*kt)/mut*cos_tht)) 
def t_s(ni, nt, ki, kt, thi, mui=4*np.pi*10**-7, mut=4*np.pi*10**-7):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)  
    return((2*(ni+1j*ki)/mui*cos_thi)/((ni+1j*ki)/mui*cos_thi+(nt+1j*kt)/mut*cos_tht))

#6: General form of fresnel coefficients for p polarized light (parrallel electric field)  
def r_p(ni, nt, ki, kt, thi, mui=4*np.pi*10**-7, mut=4*np.pi*10**-7):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    return((-(nt+1j*kt)/mut*cos_thi+(ni+1j*ki)/mui*cos_tht)/((nt+1j*kt)/mut*cos_thi+(ni+1j*ki)/mui*cos_tht))
def t_p(ni, nt, ki, kt, thi, mui=4*np.pi*10**-7, mut=4*np.pi*10**-7):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    return((2*(ni+1j*ki)/mui*cos_thi)/((nt+1j*kt)/mut*cos_thi+(ni+1j*ki)/mui*cos_tht))  


#7: transmission matrix for P poloarized light. It relates the electric field on both sides of an interface. 
def D_p(ni, nt, ki, kt, thi, mui=mu0, mut=mu0):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    return((1/t_p(ni, nt, ki, kt, thi, mui, mut))*(np.array([[1,r_p(ni, nt, ki, kt, thi, mui, mut)],
                                                             [r_p(ni, nt, ki, kt, thi, mui, mut),1]])))  

#8: transmission matrix for S poloarized light. It relates the electric field on both sides of an interface. 
def D_s(ni, nt, ki, kt, thi, mui=mu0, mut=mu0):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    return((1/t_s(ni, nt, ki, kt, thi, mui, mut))*(np.array([[1,r_s(ni, nt, ki, kt, thi, mui, mut)],
                                                             [r_s(ni, nt, ki, kt, thi, mui, mut),1]])))  


#9: propagation matrix that relates electric field on both ends of the same medium. This is independant of polarazation.
def P(ni, nt, ki, kt, thi, d, wavelength):
    cos_thi = cos(thi)
    cos_tht = costht(ni, nt, ki, kt, thi)
    return(np.array([[np.exp(1j*2*np.pi*d*(nt+1j*kt)*cos_tht/wavelength),0],
                     [0,np.exp(-1j*2*np.pi*d*(nt+1j*kt)*cos_tht/wavelength)]]))


#10: the total fresnell coefficients. Used to relate to total amount of electric field transmitted or reflected in relation to incident electric field. 
# This is currently simulating a two interface system. More can easily be added but updates will need to be made to graphing code and exell file import code. 
def t_s_tot(n0, n1, n2, k0, k1, k2, d, th0, wavelength, mui=mu0, mut=mu0):
    th1 = snells(n0, n1, k0, k1, th0)
    cos_th1 = cos(th1) 
    D_s_1 = D_s(n0, n1, k0, k1, th0, mui, mut) 
    P_s_1 = P(n0, n1, k0, k1, th0, d, wavelength) 
    D_s_2 = D_s(n1, n2, k1, k2, th1, mui, mut) 
    M = D_s_1 @ P_s_1 @ D_s_2 
    M_11 = M[0, 0]
    return(1/M_11)  
def t_p_tot(n0, n1, n2, k0, k1, k2, d, th0, wavelength, mui=mu0, mut=mu0):
    th1 = snells(n0, n1, k0, k1, th0)
    cos_th1 = cos(th1) 
    D_p_1 = D_p(n0, n1, k0, k1, th0, mui, mut) 
    P_p_1 = P(n0, n1, k0, k1, th0, d, wavelength) 
    D_p_2 = D_p(n1, n2, k1, k2, th1, mui, mut) 
    M = D_p_1 @ P_p_1 @ D_p_2 
    M_11 = M[0, 0]
    return(1/M_11)   
def r_s_tot(n0, n1, n2, k0, k1, k2, d, th0, wavelength, mui=mu0, mut=mu0):
    th1 = snells(n0, n1, k0, k1, th0)
    cos_th1 = cos(th1)
    D_s_1 = D_s(n0, n1, k0, k1, th0, mui, mut) 
    P_s_1 = P(n0, n1, k0, k1, th0, d, wavelength) 
    D_s_2 = D_s(n1, n2, k1, k2, th1, mui, mut) 
    M = D_s_1 @ P_s_1 @ D_s_2 
    M_11 = M[0, 0]
    M_21 = M[1, 0]
    return(M_21/M_11) 
def r_p_tot(n0, n1, n2, k0, k1, k2, d, th0, wavelength, mui=mu0, mut=mu0):
    th1 = snells(n0, n1, k0, k1, th0)
    cos_th1 = cos(th1)
    D_p_1 = D_p(n0, n1, k0, k1, th0, mui, mut) 
    P_p_1 = P(n0, n1, k0, k1, th0, d, wavelength) 
    D_p_2 = D_p(n1, n2, k1, k2, th1, mui, mut) 
    M = D_p_1 @ P_p_1 @ D_p_2 
    M_11 = M[0, 0] 
    M_21 = M[1, 0]
    return(M_21/M_11)

def Fresnell_rtot(polarization, i):
    n0, n1, n2 = n0_array[i], n1_array[i], n2_array[i]
    k0, k1, k2 = k0_array[i], k1_array[i], k2_array[i]
    wavelength = 1 / wave_numbers[i] # wave_number is in cm^-1 so this makes wavelength in cm which means that d needs to be in cm
    if polarization == 'p':
        return r_p_tot(n0, n1, n2, k0, k1, k2, d, th0, wavelength)#, t_p_tot(n0, n1, n2, k0, k1, k2, d, th0, wavelength)
    elif polarization=='s':
        return r_s_tot(n0, n1, n2, k0, k1, k2, d, th0, wavelength)#, t_s_tot(n0, n1, n2, k0, k1, k2, d, th0, wavelength)  
    else:
        print('you did not give a correct polarization -- enter s or p')


def reciprocal_cm_um(x):
    x = np.asarray(x, dtype=float)
    out = np.full_like(x, np.nan, dtype=float)
    np.divide(1e4, x, out=out, where=(x != 0))
    return out

In [3]:
import yaml


def set_gold_n2_k2(
    df,
    yml_path="Olmon-ev.yml",
    x_col="wavenumber",
    x_unit="cm^-1",
    n_col="n2",
    k_col="k2",
    clip=True,
):
    """
    add/overwrite gold optical constants in df[n_col] and df[k_col].

    default assumes:
        df["wavenumber"] is in cm^-1
        refractiveindex.info yaml first column is wavelength in micrometers
    """

    with open(yml_path, "r") as f:
        doc = yaml.safe_load(f)

    data_block = None
    for block in doc["DATA"]:
        if "tabulated nk" in block["type"].lower():
            data_block = block["data"]
            break

    if data_block is None:
        raise ValueError("no tabulated nk data found in yaml file")

    au = np.loadtxt(data_block.splitlines())

    table_wavelength_um = au[:, 0]
    table_n = au[:, 1]
    table_kappa = au[:, 2]

    x = df[x_col].to_numpy(dtype=float)

    if x_unit in ["cm^-1", "wavenumber"]:
        table_x = 10000.0 / table_wavelength_um
    elif x_unit in ["um", "micron", "micrometers"]:
        table_x = table_wavelength_um
    elif x_unit == "nm":
        table_x = 1000.0 * table_wavelength_um
    else:
        raise ValueError("x_unit must be 'cm^-1', 'um', or 'nm'")

    # np.interp requires the table x-values to be increasing
    order = np.argsort(table_x)
    table_x = table_x[order]
    table_n = table_n[order]
    table_kappa = table_kappa[order]

    if clip:
        x_interp = np.clip(x, table_x.min(), table_x.max())
    else:
        if x.min() < table_x.min() or x.max() > table_x.max():
            raise ValueError(
                f"df range is {x.min()} to {x.max()}, "
                f"but gold table range is {table_x.min()} to {table_x.max()}"
            )
        x_interp = x

    df[n_col] = np.interp(x_interp, table_x, table_n)
    df[k_col] = np.interp(x_interp, table_x, table_kappa)

    return df

In [4]:
#1: Here is were you upload your excel file. The path depends on your specific computer. to upload a .csv change to pd.read_csv

df = pd.read_excel("./Herguedas-SiOx-interpolated_2.xlsx", sheet_name='OG Data') # nrows avoids extraneous data at the end of each sheet
df.columns = df.columns.str.replace("p", ".", regex=False)

df['n0'] = 1
df['k0'] = 0
df = set_gold_n2_k2(
    df,
    yml_path="Olmon-ev.yml",
    x_col="wavenumber",
    x_unit="cm^-1",
)

# d is always measured in cm -- 1e-8 cm is 1 angstrom; 10e-8 cm is 1 nanometer
d = 1.65e-8  # cm

# angle is measured in degrees from the normal
th0 = 80 

# setting up other columns that can be helpful
#df['wavelength'] = 1/df['wavenumber']
#df['d'] = d*np.ones(len(df))
#df['th0'] = th0*np.ones(len(df))